[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C67_LLM_Judge_Course/04_ranking/04_ranking.ipynb)

# 04 · 从成对比较到排名（BT 拟合 / Elo 顺序依赖 / 自举区间 / 传递性 / 主动配对 / 风格控制）

目标：把一堆「A 赢 B」变成一个**带误差棒、带名次区间、带风格控制列**的排行榜。

本 notebook 你会亲手实现：
1. **Bradley–Terry 的 MLE 拟合** —— 复用模块 02 的逻辑回归，加正则化处理全胜/全败
2. **朴素胜率 vs BT** —— 对手强度不同时，朴素胜率错得有多离谱
3. **在线 Elo 的顺序依赖** —— 同一批数据换个顺序，名次会变吗
4. **自举区间与名次区间** —— 「排行榜上相差 20 分算不算差距」的定量回答
5. **传递性违反的检测** —— 三元环计数 + 分任务拟合
6. **主动配对 vs 随机配对** —— 同样预算下区间能窄多少
7. **风格控制 BT** —— 把模块 02 的去偏接进排名，输出 Δrank

> 心智模型：**看区间，不看名次。而所有这些统计工具的前提，
> 是 judge 没有共有的系统偏差——所以偏差探针永远排在排名拟合之前。**

## 1 · Bradley–Terry：拟合与基准固定

In [ ]:
import math, json, itertools
from collections import Counter, defaultdict
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -30, 30)))

def fit_bt(pairs, n_models, l2=0.01, lr=0.5, iters=4000):
    """pairs: [(i, j, y)]，y=1 表示 i 赢 j。返回中心化后的 theta。
    L2 正则处理全胜/全败导致的发散（等价于给 theta 一个高斯先验）。"""
    I = np.array([p[0] for p in pairs])
    J = np.array([p[1] for p in pairs])
    Y = np.array([p[2] for p in pairs], dtype=float)
    theta = np.zeros(n_models)
    for _ in range(iters):
        z = theta[I] - theta[J]
        p = sigmoid(z)
        err = p - Y
        grad = np.zeros(n_models)
        np.add.at(grad, I, err)
        np.add.at(grad, J, -err)
        grad = grad / len(Y) + l2 * theta
        theta -= lr * grad
    return theta - theta.mean()          # 固定基准：中心化（只有差值有意义）

def elo_from_theta(theta, base=1000.0):
    return base + (400 / math.log(10)) * np.asarray(theta)

# 造数据：8 个模型，真实强度已知
rng = np.random.default_rng(0)
N_MODELS = 8
true_theta = np.array([1.4, 1.1, 0.7, 0.3, 0.0, -0.4, -0.9, -1.5])

def simulate_pairs(true_theta, n_pairs, rng, pair_fn=None):
    n = len(true_theta)
    out = []
    for _ in range(n_pairs):
        if pair_fn is None:
            i, j = rng.choice(n, size=2, replace=False)
        else:
            i, j = pair_fn(rng)
        y = int(rng.random() < sigmoid(true_theta[i] - true_theta[j]))
        out.append((int(i), int(j), y))
    return out

pairs = simulate_pairs(true_theta, 12000, rng)
theta_hat = fit_bt(pairs, N_MODELS)
print(f"{'模型':>6}{'真实 theta':>12}{'拟合 theta':>12}{'真实 Elo':>10}{'拟合 Elo':>10}")
for k in range(N_MODELS):
    print(f'M{k:<5}{true_theta[k]:>12.3f}{theta_hat[k]:>12.3f}'
          f'{elo_from_theta(true_theta - true_theta.mean())[k]:>10.0f}'
          f'{elo_from_theta(theta_hat)[k]:>10.0f}')

corr = np.corrcoef(true_theta, theta_hat)[0, 1]
assert corr > 0.99, 'BT 应当能几乎完美地还原真实强度'
assert abs(theta_hat.mean()) < 1e-6, '必须中心化，否则绝对值没有意义'
print(f'\n拟合与真值的相关: {corr:.4f}')
print('✅ BT 就位。注意 theta 被中心化了——只有差值有意义，加一个常数所有预测都不变。')

In [ ]:
# theta 差值 → 胜率，以及 Elo 差 → 胜率的换算
print(f"{'theta 差':>10}{'胜率':>10}{'Elo 差':>10}")
for d in [0.0, 0.115, 0.3, 0.5, 1.0, 2.0]:
    print(f'{d:>10.3f}{sigmoid(d):>10.1%}{d * 400 / math.log(10):>10.0f}')

wr_20elo = sigmoid(20 * math.log(10) / 400)
print(f'\n**20 个 Elo 分 = {wr_20elo:.1%} 的胜率**')
assert abs(wr_20elo - 0.5288) < 1e-3
n_needed = math.ceil(4 * 0.25 / (wr_20elo - 0.5) ** 2)
print(f'要把 {wr_20elo:.1%} 与 50% 区分开（z=2），需要约 {n_needed:,} 场直接对局')
assert n_needed > 1000
print('✅ 而 Arena 类榜单上一对模型的直接对局往往只有几百场——')
print('   所以「相差 20 分」这件事，单靠直接对局是分辨不出来的。')

## 2 · 朴素胜率错在哪：对手强度不一样

In [ ]:
def naive_winrate(pairs, n_models):
    w = np.zeros(n_models); t = np.zeros(n_models)
    for i, j, y in pairs:
        t[i] += 1; t[j] += 1
        w[i] += y; w[j] += 1 - y
    return np.where(t > 0, w / np.maximum(t, 1), 0.5)

# 构造一个不均衡的赛程：M2 专挑弱对手，M5 专挑强对手
def biased_pairing(rng):
    r = rng.random()
    if r < 0.35:
        return 2, int(rng.integers(6, 8))       # M2 只打 M6/M7（最弱的两个）
    if r < 0.70:
        return 5, int(rng.integers(0, 2))       # M5 只打 M0/M1（最强的两个）
    i, j = rng.choice(8, size=2, replace=False)
    return int(i), int(j)

rng = np.random.default_rng(3)
pairs_b = simulate_pairs(true_theta, 16000, rng, pair_fn=biased_pairing)
naive = naive_winrate(pairs_b, N_MODELS)
theta_b = fit_bt(pairs_b, N_MODELS)

rank_true = np.argsort(-true_theta)
rank_naive = np.argsort(-naive)
rank_bt = np.argsort(-theta_b)
print(f"{'真实排名':<28}{'朴素胜率排名':<28}{'BT 排名'}")
print(f"{str([f'M{i}' for i in rank_true]):<28}"
      f"{str([f'M{i}' for i in rank_naive]):<28}"
      f"{str([f'M{i}' for i in rank_bt])}")

def spearman(a, b):
    ra = np.argsort(np.argsort(-np.asarray(a)))
    rb = np.argsort(np.argsort(-np.asarray(b)))
    ra, rb = ra - ra.mean(), rb - rb.mean()
    return float((ra * rb).sum() / math.sqrt((ra ** 2).sum() * (rb ** 2).sum()))

s_naive = spearman(true_theta, naive)
s_bt = spearman(true_theta, theta_b)
print(f'\n与真实强度的 Spearman: 朴素胜率 {s_naive:.3f} | BT {s_bt:.3f}')
assert s_bt > s_naive
assert s_bt > 0.99, 'BT 应当完全还原真实排名'
print(f'M2（真实第 3）朴素胜率 {naive[2]:.1%}（专挑最弱的两个）'
      f'→ 朴素排第 {list(rank_naive).index(2)+1} 名，BT 排第 {list(rank_bt).index(2)+1} 名')
print(f'M5（真实第 6）朴素胜率 {naive[5]:.1%}（专挑最强的两个）'
      f'→ 朴素排第 {list(rank_naive).index(5)+1} 名，BT 排第 {list(rank_bt).index(5)+1} 名')
print('\n✅ 朴素胜率把 M2 抬到了 M1 之上、把 M5 压到了 M6 之下——纯粹是赛程造成的。')
print('   BT 通过同时估计所有模型的强度，把这两个错位都修正了回来。')

## 3 · 在线 Elo 的顺序依赖：同一批数据，不同的名次

In [ ]:
def online_elo(pairs, n_models, K=0.05, init=None):
    theta = np.zeros(n_models) if init is None else np.array(init, dtype=float)
    for i, j, y in pairs:
        p = sigmoid(theta[i] - theta[j])
        upd = K * (y - p)
        theta[i] += upd
        theta[j] -= upd
    return theta - theta.mean()

rng = np.random.default_rng(7)
orders = []
for trial in range(30):
    perm = rng.permutation(len(pairs))
    shuffled = [pairs[k] for k in perm]
    th = online_elo(shuffled, N_MODELS, K=0.05)
    orders.append(np.argsort(np.argsort(-th)))     # 每个模型的名次
orders = np.array(orders)

print(f"{'模型':>6}{'名次范围':>14}{'名次标准差':>12}")
for k in range(N_MODELS):
    lo, hi = orders[:, k].min() + 1, orders[:, k].max() + 1
    print(f'M{k:<5}{f"{lo}-{hi}":>14}{orders[:, k].std():>12.2f}')

n_changed = int((orders.std(axis=0) > 0).sum())
theta_bt_fixed = fit_bt(pairs, N_MODELS)
print(f'\n30 次随机打乱顺序后，有 {n_changed}/{N_MODELS} 个模型的名次发生过变化')
assert n_changed > 0, '在线 Elo 必然存在顺序依赖'
for trial in range(3):
    perm = rng.permutation(len(pairs))
    th2 = fit_bt([pairs[k] for k in perm], N_MODELS)
    assert np.allclose(th2, theta_bt_fixed, atol=1e-6), 'BT 与顺序无关'
print('✅ 批量 BT 对顺序完全不敏感（三次打乱后拟合结果一致到 1e-6）。')
print('   → 公开榜单用在线 Elo 必须固定并公布顺序，否则不可复现。')

## 4 · 自举区间与名次区间：谁和谁其实是并列

In [ ]:
def bootstrap_bt(pairs, n_models, n_boot=400, seed=0, l2=0.01):
    """对**比较记录**重采样（不是对模型），每次重新拟合。"""
    rng = np.random.default_rng(seed)
    n = len(pairs)
    thetas = np.zeros((n_boot, n_models))
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        boot = [pairs[k] for k in idx]
        thetas[b] = fit_bt(boot, n_models, l2=l2, iters=1200)
    return thetas

pairs_small = simulate_pairs(true_theta, 2500, np.random.default_rng(11))
theta_pt = fit_bt(pairs_small, N_MODELS)
boots = bootstrap_bt(pairs_small, N_MODELS, n_boot=300, seed=1)

elo_pt = elo_from_theta(theta_pt)
elo_lo = elo_from_theta(np.percentile(boots, 2.5, axis=0))
elo_hi = elo_from_theta(np.percentile(boots, 97.5, axis=0))
ranks_boot = np.argsort(np.argsort(-boots, axis=1), axis=1) + 1
rank_lo = np.percentile(ranks_boot, 2.5, axis=0)
rank_hi = np.percentile(ranks_boot, 97.5, axis=0)

order = np.argsort(-elo_pt)
print(f"{'rank':>5}{'model':>7}{'Elo':>7}{'95% CI':>18}{'rank CI':>11}")
for r, k in enumerate(order, 1):
    print(f'{r:>5}{"M"+str(k):>7}{elo_pt[k]:>7.0f}'
          f'{f"[{elo_lo[k]:.0f}, {elo_hi[k]:.0f}]":>18}'
          f'{f"[{rank_lo[k]:.0f}, {rank_hi[k]:.0f}]":>11}')

width = float(np.mean(elo_hi - elo_lo))
print(f'\n平均区间宽度: {width:.0f} Elo 分（{len(pairs_small):,} 场比较）')
assert width > 40, '两千多场比较对应的区间不可能很窄'
overlapping = sum(1 for a, b in itertools.combinations(range(N_MODELS), 2)
                  if elo_lo[a] <= elo_hi[b] and elo_lo[b] <= elo_hi[a])
print(f'区间重叠（即「并列」）的模型对: {overlapping} / {N_MODELS*(N_MODELS-1)//2}')
assert overlapping > 0
print('✅ 相当一部分名次其实是并列——读榜规则：**看区间，不看名次**。')

In [ ]:
# 「显著优于计数」：一个天然并列友好的排序方式
def significantly_better_count(lo, hi):
    n = len(lo)
    return np.array([sum(1 for j in range(n) if j != i and lo[i] > hi[j]) for i in range(n)])

sig = significantly_better_count(elo_lo, elo_hi)
print(f"{'model':>7}{'Elo':>7}{'显著优于':>10}")
for k in np.argsort(-sig):
    print(f'{"M"+str(k):>7}{elo_pt[k]:>7.0f}{sig[k]:>10}')
assert sig.sum() < N_MODELS * (N_MODELS - 1), '不可能每一对都显著'
assert sig[np.argmax(elo_pt)] >= sig[np.argmin(elo_pt)], '最强的模型显著优于的数量应当最多'

# 样本量翻倍看这个数怎么涨
pairs_big = simulate_pairs(true_theta, 10000, np.random.default_rng(11))
boots_big = bootstrap_bt(pairs_big, N_MODELS, n_boot=200, seed=1)
lo_big = elo_from_theta(np.percentile(boots_big, 2.5, axis=0))
hi_big = elo_from_theta(np.percentile(boots_big, 97.5, axis=0))
sig_big = significantly_better_count(lo_big, hi_big)
print(f'\n显著优于计数之和: {len(pairs_small):,} 场 → {sig.sum()} | '
      f'{len(pairs_big):,} 场 → {sig_big.sum()}')
assert sig_big.sum() > sig.sum(), '样本量增大，能分辨的对更多'
print('✅ 「显著优于几个」这个数比 Elo 点估计更抗噪，也天然处理并列。')
print('   它随样本量单调上升——这条曲线本身就是「还要跑多少场」的直接指引。')

## 5 · 传递性：三元环检测与分任务拟合

In [ ]:
def cycle_rate(pairs, n_models, min_games=15):
    """三元环比例：枚举三元组，看是否形成 A>B>C>A。只统计三条边都有足够场次的三元组。"""
    wins = defaultdict(lambda: [0, 0])
    for i, j, y in pairs:
        a, b = (i, j) if i < j else (j, i)
        wins[(a, b)][0] += (y if i < j else 1 - y)
        wins[(a, b)][1] += 1
    def beats(a, b):
        key = (a, b) if a < b else (b, a)
        w, t = wins.get(key, (0, 0))
        if t < min_games:
            return None
        rate = w / t if a < b else 1 - w / t
        return rate > 0.5
    total = cyc = 0
    for a, b, c in itertools.combinations(range(n_models), 3):
        ab, bc, ca = beats(a, b), beats(b, c), beats(c, a)
        if None in (ab, bc, ca):
            continue
        total += 1
        if (ab and bc and ca) or ((not ab) and (not bc) and (not ca)):
            cyc += 1
    return (cyc / total if total else float('nan')), total

r_bt, n_tri = cycle_rate(pairs, N_MODELS)
print(f'一维真值下的三元环比例: {r_bt:.1%}（{n_tri} 个三元组）')

# 造一个二维能力的场景：两个维度，任务集混合 → 传递性会被破坏
rng = np.random.default_rng(19)
skill_2d = np.array([[1.6, -1.4], [-1.4, 1.6], [0.1, 0.1],
                     [1.0, -0.9], [-0.9, 1.0], [0.4, -0.3],
                     [-0.3, 0.4], [0.0, 0.0]])
pairs_2d = []
for _ in range(12000):
    i, j = rng.choice(N_MODELS, size=2, replace=False)
    dim = int(rng.integers(0, 2))                  # 这道题考哪个维度
    d = skill_2d[i, dim] - skill_2d[j, dim]
    pairs_2d.append((int(i), int(j), int(rng.random() < sigmoid(d))))

r_2d, n_tri2 = cycle_rate(pairs_2d, N_MODELS)
print(f'二维能力下的三元环比例: {r_2d:.1%}（{n_tri2} 个三元组）')
assert r_2d > r_bt, '能力是多维时，传递性违反显著增多'
print('\n✅ 三元环比例从一维的几个百分点涨到二维的一大截——')
print('   这不是 judge 出错，而是「一个标量」这个模型本身不够用。')

In [ ]:
# 更有信息量的检测：分任务拟合，看排名一致不一致
pairs_dim0 = [(i, j, int(np.random.default_rng(i*1000+j).random() < sigmoid(skill_2d[i,0]-skill_2d[j,0])))
              for i, j, _ in pairs_2d[:6000]]
sub0, sub1 = [], []
rng = np.random.default_rng(23)
for _ in range(8000):
    i, j = rng.choice(N_MODELS, size=2, replace=False)
    sub0.append((int(i), int(j), int(rng.random() < sigmoid(skill_2d[i,0]-skill_2d[j,0]))))
    sub1.append((int(i), int(j), int(rng.random() < sigmoid(skill_2d[i,1]-skill_2d[j,1]))))

th_all = fit_bt(pairs_2d, N_MODELS)
th0 = fit_bt(sub0, N_MODELS)
th1 = fit_bt(sub1, N_MODELS)
print(f"{'model':>7}{'总榜':>9}{'子榜A':>9}{'子榜B':>9}")
for k in range(N_MODELS):
    print(f'{"M"+str(k):>7}{th_all[k]:>9.2f}{th0[k]:>9.2f}{th1[k]:>9.2f}')

s01 = spearman(th0, th1)
top_all = int(np.argmax(th_all))
print(f'\n两个子榜之间的 Spearman: {s01:.3f}')
print(f'总榜第一是 M{top_all}；它在子榜A是第 {list(np.argsort(-th0)).index(top_all)+1} 名、'
      f'子榜B是第 {list(np.argsort(-th1)).index(top_all)+1} 名')
assert s01 < 0.5, '两个维度的排名应当显著不一致'
print('✅ 总榜第一在两个子榜上都不是第一——这不矛盾：')
print('   **总榜是各子任务按「任务集构成比例」这个隐含权重加权的结果**，而这个权重往往没人显式选过。')
print('   → 分任务榜比总榜更有决策价值。')

## 6 · 主动配对：同样预算，区间能窄多少

In [ ]:
def pair_info(theta, i, j):
    """单场比较的期望信息量：预测胜率的二元熵。"""
    p = float(sigmoid(theta[i] - theta[j]))
    if p <= 0 or p >= 1:
        return 0.0
    return -(p * math.log2(p) + (1 - p) * math.log2(1 - p))

def collect(strategy, n_rounds, true_theta, seed=0, eps=0.15, batch=50):
    """strategy: 'random' | 'active'。每 batch 场重新拟合一次当前估计。"""
    rng = np.random.default_rng(seed)
    n = len(true_theta)
    got, theta_cur = [], np.zeros(n)
    for r in range(n_rounds // batch):
        for _ in range(batch):
            if strategy == 'random' or rng.random() < eps:
                i, j = rng.choice(n, size=2, replace=False)
            else:
                cands = [(pair_info(theta_cur, a, b), a, b)
                         for a, b in itertools.combinations(range(n), 2)]
                _, i, j = max(cands, key=lambda t: t[0] + rng.normal(0, 0.02))
            y = int(rng.random() < sigmoid(true_theta[i] - true_theta[j]))
            got.append((int(i), int(j), y))
        theta_cur = fit_bt(got, n, iters=800)
    return got

def adjacent_diff_width(boots, true_theta):
    # 相邻名次之间「强度差」的自举区间宽度——这才是主动配对真正优化的量
    order = np.argsort(-np.asarray(true_theta))
    widths = []
    for a, b in zip(order, order[1:]):
        d = boots[:, a] - boots[:, b]
        widths.append(np.percentile(d, 97.5) - np.percentile(d, 2.5))
    return float(np.mean(widths))

def distinguishable_pairs(lo, hi):
    n = len(lo)
    return sum(1 for i, j in itertools.combinations(range(n), 2)
               if lo[i] > hi[j] or lo[j] > hi[i])

BUDGET = 3000
res = {}
print(f"{'策略':<8}{'平均区间宽度':>14}{'相邻差区间宽度':>16}{'可区分的对':>12}")
for strat in ['random', 'active']:
    data = collect(strat, BUDGET, true_theta, seed=5)
    bo = bootstrap_bt(data, N_MODELS, n_boot=200, seed=2)
    lo, hi = np.percentile(bo, 2.5, axis=0), np.percentile(bo, 97.5, axis=0)
    elo_lo_, elo_hi_ = elo_from_theta(lo), elo_from_theta(hi)
    w_mean = float(np.mean(elo_hi_ - elo_lo_))
    w_adj = adjacent_diff_width(bo, true_theta)
    n_dist = distinguishable_pairs(elo_lo_, elo_hi_)
    res[strat] = dict(w_mean=w_mean, w_adj=w_adj, n_dist=n_dist, data=data)
    print(f'{strat:<8}{w_mean:>14.1f}{w_adj:>16.3f}{n_dist:>12}')

assert res['active']['w_adj'] < res['random']['w_adj'], '主动配对应当让相邻名次更容易分开'
print(f'\n相邻名次的强度差区间: 主动配对窄了 '
      f'{1 - res["active"]["w_adj"]/res["random"]["w_adj"]:.0%}')
assert res['active']['n_dist'] < res['random']['n_dist']
print(f'但「可区分的模型对」总数反而少了: {res["random"]["n_dist"]} → {res["active"]["n_dist"]}')

# 原因：主动配对下，强弱悬殊的对被比得更少
def pair_counts(data, n):
    c = np.zeros((n, n))
    for i, j, _ in data:
        c[i, j] += 1; c[j, i] += 1
    return c
c_rand = pair_counts(res['random']['data'], N_MODELS)
c_act = pair_counts(res['active']['data'], N_MODELS)
far = abs(true_theta[0] - true_theta[-1])
print(f'\n最强 vs 最弱（theta 差 {far:.1f}）的直接对局数: '
      f'随机 {c_rand[0,-1]:.0f} 场 → 主动 {c_act[0,-1]:.0f} 场')
assert c_act[0, -1] < c_rand[0, -1]
print('\n✅ 这是本节最诚实的结论：**主动配对不是「全面更好」，而是「把精度搬了个地方」。**')
print('   它把预算从「结果已知的悬殊对局」搬到「难分胜负的相邻名次」上——')
print('   相邻名次因此更容易分开，但悬殊对的估计变得完全依赖模型外推，')
print('   而那恰恰是传递性假设最可能失效的地方。所以要保留 eps≥0.1 的随机配对做覆盖。')

## 7 · 风格控制 BT：把去偏接进排名

In [ ]:
def fit_bt_with_covariates(pairs, X, n_models, l2=0.01, l2_gamma=0.0,
                           lr=1.0, iters=20000):
    """logit P(i>j) = (theta_i - theta_j) + gamma^T (x_i - x_j)。
    X: (n_models, d) 每个模型的平均风格特征（已标准化）。返回 (theta, gamma)。
    注意 gamma 用单独（更弱的）正则：它是**一个全局参数**，
    对它做和 theta 同样强度的收缩会系统性低估风格效应。"""
    I = np.array([p[0] for p in pairs]); J = np.array([p[1] for p in pairs])
    Y = np.array([p[2] for p in pairs], dtype=float)
    X = np.asarray(X, dtype=float)
    d = X.shape[1]
    theta = np.zeros(n_models); gamma = np.zeros(d)
    DX = X[I] - X[J]
    for _ in range(iters):
        z = theta[I] - theta[J] + DX @ gamma
        err = sigmoid(z) - Y
        g_theta = np.zeros(n_models)
        np.add.at(g_theta, I, err); np.add.at(g_theta, J, -err)
        theta -= lr * (g_theta / len(Y) + l2 * theta)
        gamma -= lr * (DX.T @ err / len(Y) + l2_gamma * gamma)
    return theta - theta.mean(), gamma

# 造数据：真实质量 + 风格。judge 对风格有偏好（gamma_true > 0）
rng = np.random.default_rng(29)
quality = np.array([1.2, 0.9, 0.6, 0.3, 0.0, -0.3, -0.7, -1.2])
style = np.array([[-0.8], [1.5], [0.2], [-1.0], [0.0], [1.2], [-0.4], [0.6]])  # 标准化后的"长度/排版"
GAMMA_TRUE = 0.8

pairs_style = []
for _ in range(20000):
    i, j = rng.choice(N_MODELS, size=2, replace=False)
    z = (quality[i] - quality[j]) + GAMMA_TRUE * (style[i, 0] - style[j, 0])
    pairs_style.append((int(i), int(j), int(rng.random() < sigmoid(z))))

th_raw = fit_bt(pairs_style, N_MODELS)
th_sc, gamma_hat = fit_bt_with_covariates(pairs_style, style, N_MODELS)
print(f'拟合的风格系数 gamma = {gamma_hat[0]:.3f}（真值 {GAMMA_TRUE}）')
assert abs(gamma_hat[0] - GAMMA_TRUE) < 0.2

rank_raw = np.argsort(np.argsort(-th_raw)) + 1
rank_sc = np.argsort(np.argsort(-th_sc)) + 1
print(f"\n{'model':>7}{'原始 Elo':>11}{'风格控制 Elo':>15}{'原名次':>8}{'控制后':>8}{'Δrank':>8}{'风格':>8}")
for k in np.argsort(rank_raw):
    print(f'{"M"+str(k):>7}{elo_from_theta(th_raw)[k]:>11.0f}'
          f'{elo_from_theta(th_sc)[k]:>15.0f}{rank_raw[k]:>8}{rank_sc[k]:>8}'
          f'{rank_raw[k]-rank_sc[k]:>+8}{style[k,0]:>8.1f}')

corr_sc = np.corrcoef(quality, th_sc)[0, 1]
corr_raw = np.corrcoef(quality, th_raw)[0, 1]
assert corr_sc > corr_raw, '风格控制后应当更接近真实质量'
print(f'\n与真实质量的相关: 原始 {corr_raw:.3f} → 风格控制后 {corr_sc:.3f}')
print('✅ Δrank 这一列是最有信息量的输出：')
print('   风格分高的模型（M1、M5）控制后掉名次，风格分低的（M0、M3）升名次。')
print('   这未必是坏事（如果用户确实喜欢那种风格），但它必须被知道，而不是混在一个总分里。')

## ✏️ 练习 1：Elo 差与所需场次

实现 `elo_to_winrate(elo_diff)` 与 `games_for_elo_gap(elo_diff, target_z=2.0)`：
前者把 Elo 差换算成胜率（$\sigma(\Delta \ln 10 / 400)$），
后者返回把该胜率与 50% 区分开所需的场次（$z^2 \cdot 0.25 / (p-0.5)^2$，向上取整）。

In [ ]:
def elo_to_winrate(elo_diff):
    # TODO
    raise NotImplementedError

def games_for_elo_gap(elo_diff, target_z=2.0):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert abs(elo_to_winrate(0) - 0.5) < 1e-12
assert abs(elo_to_winrate(400) - 10 / 11) < 1e-9      # 差 400 分 = 胜率 10:1
assert abs(elo_to_winrate(20) - 0.5288) < 1e-3
assert games_for_elo_gap(20) > games_for_elo_gap(100)
print(f"{'Elo 差':>8}{'胜率':>10}{'所需场次':>12}")
for d in [10, 20, 50, 100, 200]:
    print(f'{d:>8}{elo_to_winrate(d):>10.1%}{games_for_elo_gap(d):>12,}')
print('✅ 练习 1 通过：这张表就是「读榜时相差多少分才算真差距」的直接答案。')

## ✏️ 练习 2：名次区间与并列判定

实现 `rank_intervals(boot_thetas)`：输入自举得到的 `(n_boot, n_models)` 参数矩阵，
返回每个模型的 `(点估计名次, 2.5% 分位名次, 97.5% 分位名次)`。
再实现 `tied_groups(lo, hi)`：返回所有「名次区间重叠」的模型对集合。

In [ ]:
def rank_intervals(boot_thetas):
    # TODO：名次 = 按 theta 降序的位次（从 1 开始）
    raise NotImplementedError

def tied_groups(lo, hi):
    # TODO：返回 [(i, j), ...]，i < j 且区间重叠
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
pt_r, lo_r, hi_r = rank_intervals(boots)
assert len(pt_r) == N_MODELS
assert all(lo_r[k] <= pt_r[k] <= hi_r[k] for k in range(N_MODELS))
ties = tied_groups(lo_r, hi_r)
assert all(i < j for i, j in ties)
print(f"{'model':>7}{'名次':>7}{'名次区间':>12}")
for k in np.argsort(pt_r):
    print(f'{"M"+str(k):>7}{pt_r[k]:>7.0f}{f"[{lo_r[k]:.0f}, {hi_r[k]:.0f}]":>12}')
print(f'\n判定为并列的模型对: {[(f"M{i}", f"M{j}") for i, j in ties]}')
assert len(ties) > 0, '2500 场比较下必然有并列'
print('✅ 练习 2 通过：把「名次」报成区间，读者就不会去解读那些本来就分不开的差距。')

## ✏️ 练习 3：比较图的连通性检查

实现 `is_connected(pairs, n_models, min_games=1)`：
把「比过至少 `min_games` 场」的模型对视为一条边，判断整个图是否连通（BFS 即可）。
不连通时 BT 参数在组之间不可识别。

In [ ]:
def is_connected(pairs, n_models, min_games=1):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert is_connected(pairs, N_MODELS) is True
# 构造一个断裂的赛程：M0-M3 一组，M4-M7 一组，两组从不交手
rng = np.random.default_rng(41)
split_pairs = []
for _ in range(3000):
    grp = [0, 1, 2, 3] if rng.random() < 0.5 else [4, 5, 6, 7]
    i, j = rng.choice(grp, size=2, replace=False)
    split_pairs.append((int(i), int(j), int(rng.random() < sigmoid(true_theta[i]-true_theta[j]))))
assert is_connected(split_pairs, N_MODELS) is False
assert is_connected(pairs, N_MODELS, min_games=100000) is False   # 阈值太高 → 无边
th_split = fit_bt(split_pairs, N_MODELS)
print('断裂赛程下拟合出的 theta:', np.round(th_split, 2))
print('组内 Spearman（M0-M3）:', round(spearman(true_theta[:4], th_split[:4]), 3))
print('✅ 练习 3 通过：组内排序仍然对，但**跨组的相对位置完全由正则化决定，没有数据支撑**。')
print('   所以比较图的连通性必须写进报告——不连通时跨组比较毫无意义。')

## ✏️ 练习 4：风格贡献度

实现 `style_contribution(theta_raw, theta_sc)`：返回每个模型
`(原始 Elo, 控制后 Elo, 风格贡献的 Elo 分数, 名次变化)`，
风格贡献 = 原始 Elo − 控制后 Elo。

In [ ]:
def style_contribution(theta_raw, theta_sc):
    # TODO：返回一个 list，每项是 (elo_raw, elo_sc, style_elo, delta_rank)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
contrib = style_contribution(th_raw, th_sc)
assert len(contrib) == N_MODELS
style_elos = np.array([c[2] for c in contrib])
assert np.corrcoef(style_elos, style[:, 0])[0, 1] > 0.9, '风格贡献必须与风格特征强相关'
assert abs(sum(c[3] for c in contrib)) < 1e-9, '名次变化之和必然为 0'
print(f"{'model':>7}{'原始':>9}{'控制后':>9}{'风格贡献':>10}{'Δrank':>8}")
for k in range(N_MODELS):
    r, sc_, st, dr = contrib[k]
    print(f'{"M"+str(k):>7}{r:>9.0f}{sc_:>9.0f}{st:>+10.0f}{dr:>+8}')
print('✅ 练习 4 通过：「风格贡献了多少 Elo 分」是一个可以直接放进榜单的列。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def elo_to_winrate(elo_diff):
    return float(sigmoid(elo_diff * math.log(10) / 400))

def games_for_elo_gap(elo_diff, target_z=2.0):
    p = elo_to_winrate(elo_diff)
    if abs(p - 0.5) < 1e-12:
        return float('inf')
    return math.ceil(target_z ** 2 * 0.25 / (p - 0.5) ** 2)

In [ ]:
# 练习 2 参考答案
def rank_intervals(boot_thetas):
    B = np.asarray(boot_thetas)
    ranks = np.argsort(np.argsort(-B, axis=1), axis=1) + 1
    pt = np.median(ranks, axis=0)
    lo = np.percentile(ranks, 2.5, axis=0)
    hi = np.percentile(ranks, 97.5, axis=0)
    return pt, lo, hi

def tied_groups(lo, hi):
    n = len(lo)
    return [(i, j) for i, j in itertools.combinations(range(n), 2)
            if lo[i] <= hi[j] and lo[j] <= hi[i]]

In [ ]:
# 练习 3 参考答案
def is_connected(pairs, n_models, min_games=1):
    cnt = defaultdict(int)
    for i, j, _ in pairs:
        a, b = (i, j) if i < j else (j, i)
        cnt[(a, b)] += 1
    adj = defaultdict(set)
    for (a, b), c in cnt.items():
        if c >= min_games:
            adj[a].add(b); adj[b].add(a)
    seen, stack = {0}, [0]
    while stack:
        u = stack.pop()
        for v in adj[u]:
            if v not in seen:
                seen.add(v); stack.append(v)
    return len(seen) == n_models

In [ ]:
# 练习 4 参考答案
def style_contribution(theta_raw, theta_sc):
    e_raw = elo_from_theta(theta_raw)
    e_sc = elo_from_theta(theta_sc)
    r_raw = np.argsort(np.argsort(-np.asarray(theta_raw))) + 1
    r_sc = np.argsort(np.argsort(-np.asarray(theta_sc))) + 1
    return [(float(e_raw[k]), float(e_sc[k]), float(e_raw[k] - e_sc[k]),
             int(r_raw[k] - r_sc[k])) for k in range(len(theta_raw))]

---
## 🧪 真实工程胶囊：排行榜的落地

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════
# A. 从 judge 结果表到排行榜（完整流程）
# ══════════════════════════════════════════════════════════════════
import pandas as pd, numpy as np
df = pd.read_json("judge_results.jsonl", lines=True)
# 必需列: model_a, model_b, verdict(A/B/tie), a_len_tokens, b_len_tokens, order, consistent

# 1) 只用 swap 一致的样本；不一致的记平局（模块 01/02）
df = df[df.consistent | (df.verdict == "tie")]

# 2) 平局按半分拆成两条记录（与 BT 的处理一致）
rows = []
for r in df.itertuples():
    if r.verdict == "tie":
        rows += [(r.model_a, r.model_b, 1, 0.5), (r.model_a, r.model_b, 0, 0.5)]
    else:
        rows.append((r.model_a, r.model_b, int(r.verdict == "A"), 1.0))
# 注：带权重的 BT 需要在梯度里乘以 weight；简化做法是把平局拆成两条等权记录。

# 3) 连通性检查 —— 不连通时直接停止，不要出榜
assert is_connected(pairs, n_models), "比较图不连通，跨组排名无意义"

# 4) 拟合 + 自举 + 名次区间
theta = fit_bt(pairs, n_models, l2=0.01)
boots = bootstrap_bt(pairs, n_models, n_boot=1000)
pt, lo, hi = rank_intervals(boots)

# 5) 风格控制版：X 用每个模型的**平均**风格特征（标准化后）
X = np.column_stack([
    zscore(df.groupby("model").a_len_tokens.mean()),
    zscore(df.groupby("model").md_heading_count.mean()),
    zscore(df.groupby("model").list_item_count.mean()),
])
theta_sc, gamma = fit_bt_with_covariates(pairs, X, n_models)

# ══════════════════════════════════════════════════════════════════
# B. 必须写进榜单页脚的六行
# ══════════════════════════════════════════════════════════════════
# judge 模型 + prompt 哈希 + 是否开 swap
# 拟合方法 + 正则化强度 + 基准如何固定
# 区间方法（对**比较记录**自举）+ 自举次数
# 比较总数 + 模型数 + 比较图连通性
# 配对策略（随机 / 主动 / 混合比例）
# 传递性检查结果 + 子榜之间的 Spearman

# ══════════════════════════════════════════════════════════════════
# C. 增量更新：什么时候可以不重拟合
# ══════════════════════════════════════════════════════════════════
# 新增比较 < 总量的 5%  → 用在线 Elo 做预览，正式榜单仍走批量 BT
# 新增了一个模型        → 必须重拟合（新模型会改变所有人的相对位置）
# 换了 judge / prompt   → **所有历史比较作废**，不能与新数据混拟合
'''
print(RECIPE)

### 小结

| 你学到的 | 一句话 | 用在哪 |
|---|---|---|
| BT vs 朴素胜率 | 朴素胜率被赛程污染；BT 同时估计所有强度 | 任何排行榜 |
| theta 与 Elo | Elo 是 BT 的仿射变换；20 Elo 分 = 52.9% 胜率 | 读榜心算 |
| Elo 顺序依赖 | 在线 Elo 的分数是「数据 + 处理顺序」的函数 | 选拟合方法 |
| 自举与名次区间 | 看区间不看名次；区间重叠即并列 | 报告规范 |
| 传递性 | 总榜第一在子榜上常常不是第一，这不矛盾 | 优先看子榜 |
| 主动配对 | 区间更窄，但强弱悬殊的对被比得更少，要留 ε 随机 | 预算分配 |
| 风格控制 BT | Δrank 是最有信息量的一列 | 诊断风格贡献 |

下一模块：**05 · 奖励模型与过优化**——judge 变成训练信号之后会发生什么，
RewardBench 类评测怎么做，以及 KL–奖励曲线上那个必然出现的拐点。